In [4]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_curve, auc, roc_auc_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

def evaluate_with_meta_model(
    ruleset_path,
    train_correct_path,
    train_misclassified_path,
    test_data_folder,
    output_dir,
    use_stratified_sampling=False  # <-- New optional parameter
):
    """
    Trains a meta-model on stylistic features to predict classification correctness
    and evaluates it on unseen data using AUC-ROC.
    """
    print(f"[1] Loading ruleset from: {ruleset_path}")
    ruleset_df = pd.read_csv(ruleset_path)
    feature_cols = ruleset_df['feature'].unique().tolist()
    if not feature_cols:
        print("  [ERROR] No features found in the ruleset file. Skipping.")
        return

    # --- Step 2: Prepare Training Data for the Meta-Model ---
    print("[2] Preparing training data for the meta-model...")
    df_train_cor = pd.read_csv(train_correct_path)
    df_train_mis = pd.read_csv(train_misclassified_path)

    # --- Optional Stratified Sampling Block ---
    if use_stratified_sampling:
        print("  >>> Balancing training data with stratified sampling...")
        mis_counts = df_train_mis['stance'].value_counts()
        df_train_cor_sampled = (
            df_train_cor.groupby('stance')
            .apply(lambda x: x.sample(
                n=min(len(x), mis_counts.get(x.name, 0)),
                random_state=42
            ))
            .reset_index(drop=True)
        )
        df_train_cor_sampled['meta_target'] = 1
        df_train_mis['meta_target'] = 0
        df_train = pd.concat([df_train_cor_sampled, df_train_mis], ignore_index=True)
        print(f"  Using {len(df_train_cor_sampled)} correct and {len(df_train_mis)} misclassified samples for training.")
    else:
        print("  >>> Using full, imbalanced training data.")
        df_train_cor['meta_target'] = 1
        df_train_mis['meta_target'] = 0
        df_train = pd.concat([df_train_cor, df_train_mis], ignore_index=True)
    # --- End of Optional Block ---

    for col in feature_cols:
        if col not in df_train.columns:
            df_train[col] = np.nan

    X_train = df_train[feature_cols]
    y_train = df_train['meta_target']

    imputer = SimpleImputer(strategy='median')
    X_train_imputed = imputer.fit_transform(X_train)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)

    # --- Step 3: Train the Meta-Model ---
    print("[3] Training the Logistic Regression meta-model...")
    meta_model = LogisticRegression(penalty='l2', random_state=42, max_iter=1000)
    meta_model.fit(X_train_scaled, y_train)

    # --- Step 4: Prepare Test Data (Test data is never sampled) ---
    print("[4] Preparing unseen test data for evaluation...")
    df_test = pd.read_csv(os.path.join(test_data_folder, "wtwt_test_processed.csv"))
    corr_idx = np.load(os.path.join(test_data_folder, "correctly_classified_indices.npy"))
    mis_idx = np.load(os.path.join(test_data_folder, "misclassified_indices.npy"))
    df_test['actual_outcome_binary'] = 0
    df_test.loc[corr_idx, 'actual_outcome_binary'] = 1

    X_test = df_test[feature_cols]
    y_test = df_test['actual_outcome_binary']
    X_test_imputed = imputer.transform(X_test)
    X_test_scaled = scaler.transform(X_test_imputed)

    # --- Step 5: Evaluate the Meta-Model ---
    print("[5] Evaluating meta-model on test data...")
    y_pred_proba = meta_model.predict_proba(X_test_scaled)[:, 1]
    y_pred = meta_model.predict(X_test_scaled)

    # Generate the classification report
    report = classification_report(
        y_test,
        y_pred,
        target_names=['Misclassified', 'Correct'], # Labels for 0 and 1
        digits=4
    )
    
    # Print the report to the console
    print("\n  >>> Classification Report <<<")
    print(report)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    print(f"\n  >>> Meta-Model Performance <<<")
    print(f"  AUC Score: {auc_score:.4f}")

    # --- Step 6: Generate Outputs ---
    print("[6] Generating ROC curve plot and saving results...")
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Meta-Model ROC Curve for Predicting Correctness')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()

    roc_plot_path = os.path.join(output_dir, "meta_model_roc_curve.png")
    plt.savefig(roc_plot_path, dpi=300)
    print(f"  ROC curve plot saved to ➜ {roc_plot_path}")
    plt.close()

    results_df = pd.DataFrame({'true_outcome': y_test, 'predicted_correctness_probability': y_pred_proba})
    results_csv_path = os.path.join(output_dir, "meta_model_results.csv")
    results_df.to_csv(results_csv_path, index=False)
    print(f"  Evaluation results saved to ➜ {results_csv_path}")

    return results_df

if __name__ == '__main__':
    sns.set_theme(style="whitegrid")

    # --- Define Experiments ---
    experiments_to_run = [
        {'coef': 'positive', 'sig': 'yes', 'sampling': True},
        {'coef': 'negative', 'sig': 'yes', 'sampling': True},
    ]
    coef_thresholds = np.round(np.arange(0.10, 1.00, 0.10), 2).tolist()
    
    for threshold in coef_thresholds:
        # Define paths
        BASE_PLOTS_DIR = 'balanced_analysis'
        EVALUATION_DATA_FOLDER = 'evaluation_data'
        ORIGINAL_CORRECT_DATA_PATH = '/home/p/parush/style_markers/classifications/whole/correctly_classified_examples_processed.csv'
        ORIGINAL_MISCLASSIFIED_DATA_PATH = '/home/p/parush/style_markers/classifications/whole/misclassified_examples_processed.csv'
        
        for config in experiments_to_run:
            coef_direction = config['coef']
            significance_filter = config['sig']
            use_sampling = config['sampling']

            print(f"\n{'='*60}")
            print(f"RUNNING EVALUATION FOR: Threshold={threshold}, Coef='{coef_direction}', Sig='{significance_filter}', Balanced={use_sampling}")
            print(f"{'='*60}\n")

            # --- Define experiment-specific paths ---
            experiment_name = f"whole_whole_coef-{coef_direction}_sig-{significance_filter}"
            if use_sampling:
                experiment_name += "_balanced"
            
            experiment_folder = os.path.join(BASE_PLOTS_DIR, f"{threshold}_threshold_experiment", experiment_name)
            ruleset_path = os.path.join(experiment_folder, "filtered_features_summary.csv")
            evaluation_output_dir = os.path.join(experiment_folder, "meta_model_evaluation")
            os.makedirs(evaluation_output_dir, exist_ok=True)

            if not os.path.exists(ruleset_path):
                print(f"  [ERROR] Ruleset file not found, skipping: {ruleset_path}")
                continue

            # Call the evaluation function with the correct sampling flag
            evaluate_with_meta_model(
                ruleset_path=ruleset_path,
                train_correct_path=ORIGINAL_CORRECT_DATA_PATH,
                train_misclassified_path=ORIGINAL_MISCLASSIFIED_DATA_PATH,
                test_data_folder=EVALUATION_DATA_FOLDER,
                output_dir=evaluation_output_dir,
                use_stratified_sampling=use_sampling  # <-- Pass the flag
            )

    print("\n\nAll evaluations complete.")


RUNNING EVALUATION FOR: Threshold=0.1, Coef='positive', Sig='yes', Balanced=True

[1] Loading ruleset from: balanced_analysis/0.1_threshold_experiment/whole_whole_coef-positive_sig-yes_balanced/filtered_features_summary.csv
[2] Preparing training data for the meta-model...
  >>> Balancing training data with stratified sampling...
  Using 275 correct and 275 misclassified samples for training.
[3] Training the Logistic Regression meta-model...
[4] Preparing unseen test data for evaluation...
[5] Evaluating meta-model on test data...

  >>> Classification Report <<<
               precision    recall  f1-score   support

Misclassified     0.2350    0.2796    0.2554      2679
      Correct     0.7555    0.7098    0.7320      8402

     accuracy                         0.6058     11081
    macro avg     0.4953    0.4947    0.4937     11081
 weighted avg     0.6297    0.6058    0.6167     11081


  >>> Meta-Model Performance <<<
  AUC Score: 0.4992
[6] Generating ROC curve plot and saving 